# Forecasting Analysis

This notebook reproduces the forecasting workflow for the retail dataset and mirrors the logic in `src/forecasting.py`.

It develops short-term forecasts for sales performance (`daily_revenue`) and order volumes (`daily_order_volume`) using two approaches:
- Seasonal Naive
- Additive Holt-Winters

The lower-RMSE model is used for the forward 30-day forecast.

In [7]:
from pathlib import Path
import sys

root = Path.cwd()

if root.name == "notebooks":
    root = root.parent

sys.path.append(str(root / "src"))

from forecasting import (
    DATA_PATH,
    FORECAST_DAYS,
    TEST_DAYS,
    evaluate_target,
    load_daily_targets,
    render_table,
    summarize_forecast,
)

## Load And Prepare Daily Targets

The cleaned transactional dataset is aggregated into two daily time series:
- `daily_revenue`
- `daily_order_volume`

Missing dates are kept and treated as zero-activity days so the model can learn the weekly operating pattern.

In [2]:
dates, targets = load_daily_targets(DATA_PATH)

print(f"Observations: {len(dates)} daily records")
print(f"Date range: {dates[0].isoformat()} to {dates[-1].isoformat()}")
print("Targets:", ", ".join(targets.keys()))
print(f"Holdout window: last {TEST_DAYS} days")
print(f"Forecast horizon: next {FORECAST_DAYS} days")

Observations: 374 daily records
Date range: 2010-12-01 to 2011-12-09
Targets: daily_revenue, daily_order_volume
Holdout window: last 30 days
Forecast horizon: next 30 days


## Evaluate Competing Forecast Models

Each target is split chronologically:
- training set: all observations except the final 30 days
- test set: the final 30 days

The notebook scores both models with MAE, RMSE, and WAPE.

`Accuracy` is not used here because both targets are continuous regression outputs rather than class labels.

In [3]:
all_metrics = []
all_forecasts = []
all_summaries = []

for target_name, series in targets.items():
    target_metrics, target_forecasts = evaluate_target(target_name, series, dates)
    all_metrics.extend(target_metrics)
    all_forecasts.extend(target_forecasts)
    all_summaries.append(
        summarize_forecast(
            target_name,
            target_forecasts[0]["model"],
            [row["forecast"] for row in target_forecasts],
        )
    )

print("Model evaluation:")
print(render_table(all_metrics, ["target", "model", "mae", "rmse", "wape"]))

Model evaluation:
target              model           mae       rmse      wape 
------------------  --------------  --------  --------  -----
daily_revenue       Seasonal Naive  5822.61   8048.46   27.42
daily_revenue       Holt-Winters    13193.88  15816.71  62.14
daily_order_volume  Seasonal Naive  14.70     19.06     17.84
daily_order_volume  Holt-Winters    56.81     67.30     68.94


## Forecast Summary

The forward forecast uses the lower-RMSE model for each target. In the current dataset, Seasonal Naive is selected because it outperforms Holt-Winters on both revenue and order volume.

In [4]:
print("30-day forecast summary:")
print(
    render_table(
        all_summaries,
        [
            "target",
            "selected_model",
            "30_day_total_forecast",
            "average_daily_forecast",
            "peak_day_forecast",
        ],
    )
)

preview_rows = []
seen_by_target = {}
for row in all_forecasts:
    seen_by_target.setdefault(row["target"], 0)
    if seen_by_target[row["target"]] < 7:
        preview_rows.append(row)
        seen_by_target[row["target"]] += 1

print()
print("Next 7 forecasted days:")
print(render_table(preview_rows, ["date", "target", "model", "forecast"]))

30-day forecast summary:
target              selected_model  30_day_total_forecast  average_daily_forecast  peak_day_forecast
------------------  --------------  ---------------------  ----------------------  -----------------
daily_revenue       Seasonal Naive  526053.01              17535.10                32482.45         
daily_order_volume  Seasonal Naive  2090.00                69.67                   109.00           

Next 7 forecasted days:
date        target              model           forecast
----------  ------------------  --------------  --------
2011-12-10  daily_revenue       Seasonal Naive  0.00    
2011-12-11  daily_revenue       Seasonal Naive  14558.65
2011-12-12  daily_revenue       Seasonal Naive  32482.45
2011-12-13  daily_revenue       Seasonal Naive  22549.66
2011-12-14  daily_revenue       Seasonal Naive  25878.46
2011-12-15  daily_revenue       Seasonal Naive  23294.46
2011-12-16  daily_revenue       Seasonal Naive  9109.91 
2011-12-10  daily_order_volume  S

## Forecasted Trends

The 30-day outlook points to a stable weekly operating cycle rather than a strong long-run upward or downward shift:
- projected sales performance is `526053.01` in revenue over the next 30 days, with an average of `17535.10` per day
- projected order volume is `2090` orders over the next 30 days, with an average of `69.67` orders per day
- the next 7 forecasted days repeat the recent weekly pattern, including a zero-activity Saturday followed by stronger Sunday-to-Thursday demand

This makes the forecast operationally useful because it identifies when capacity pressure is likely to rise and when lower-demand days can absorb maintenance, replenishment, or lighter staffing.

## Operational Impact

Short-term forecasts from this notebook can be used to:
- plan staffing for order-processing peaks
- prepare replenishment for higher-demand days
- estimate near-term revenue for tactical cash-flow decisions

The main limitation is that the model only sees transaction history. Promotions, holidays, and supply disruptions are not included, so the forecasts should be used as tactical guidance rather than a long-range plan.